<a href="https://colab.research.google.com/github/AlesioRios/Laboratorio-de-Datos-II/blob/main/Lab_de_Datos_II_TP3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Descargar el archivo worldbank_dataset.csv
#!gdown --id 1artALqi9at8DqzTp5x8NA7P1qwJLKN24

In [6]:
import pandas as pd

df = pd.read_csv('/content/worldbank_dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/worldbank_dataset.csv'

# Ejercicio 1.1 — Análisis exploratorio inicial

EDA de diagnóstico antes de cualquier tratamiento de faltantes, transformación o estandarización (eso es Ejercicio 1, puntos 2 a 4, y queda para después). El objetivo acá es entender qué hay: nulos, rango/orden de magnitud de cada variable, y forma de las distribuciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
from IPython.display import display, HTML


def estilizar(df, cmap='Blues', precision=2):
  # Codifica una tabla con color en vez de imprimirla como texto plano (heatmap-style, por columna).
  # Ademas de background_gradient, se agregan estilos de tabla explicitos: al extraer el
  # HTML con .to_html() para mostrarlo dentro de un <div> propio (mostrar_lado_a_lado),
  # se pierde el CSS por default que Colab/Jupyter le pone a un DataFrame (padding, bordes,
  # tipografia) - sin esto se ve mas apretado/plano que una tabla mostrada normalmente.
  estilos_tabla = [
      {'selector': 'th', 'props': [('padding', '6px 12px'), ('text-align', 'center'), ('border', '1px solid #d0d7de'), ('background-color', '#f6f8fa')]},
      {'selector': 'td', 'props': [('padding', '6px 12px'), ('text-align', 'center'), ('border', '1px solid #d0d7de')]},
      {'selector': ''  , 'props': [('border-collapse', 'collapse'), ('font-size', '13px')]},
  ]
  return (
      df.style
      .background_gradient(cmap=cmap, axis=0)
      .format(precision=precision)
      .set_table_styles(estilos_tabla)
  )


def mostrar_lado_a_lado(tablas, cmap='Blues', precision=2):
  # Muestra varias tablas en una fila horizontal (en vez de apiladas) para que entren juntas en pantalla.
  # Cada tabla queda dentro de una tarjeta con borde propio, con mas separacion entre ellas
  # y entre el titulo y la tabla, para que se vea prolijo en vez de amontonado.
  bloques = []
  for titulo, df in tablas:
    bloques.append(
        "<div style='margin:0 32px 20px 0; padding:12px 14px; "
        "border:1px solid #d0d7de; border-radius:10px;'>"
        f"<div style='font-weight:600; margin-bottom:8px;'>{titulo}</div>"
        f"{estilizar(df, cmap=cmap, precision=precision).to_html()}</div>"
    )
  display(HTML("<div style='display:flex; flex-wrap:wrap;'>" + ''.join(bloques) + "</div>"))


# Variable objetivo de todo el TP3 (problema de regresion)
TARGET = 'gdp_per_capita_current_us$'

# country/iso_code/countryiso3 son tres codificaciones del mismo pais (misma cardinalidad,
# 144 cada una) - se excluyen del analisis numerico para no analizar tres veces lo mismo.
COLUMNAS_ID = ['country', 'iso_code', 'countryiso3']
columnas_numericas = [c for c in df.columns if c not in COLUMNAS_ID and c != 'year']

### 1. Estructura del dataset

¿Con cuántos datos contamos y de qué tipo es cada columna?

In [ ]:
print(f"Dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Paises: {df['country'].nunique()} | Anios: {df['year'].min()}-{df['year'].max()} (panel pais-anio)")

display(df.dtypes.value_counts().rename('Cantidad de columnas').to_frame())
df.head()

### Conclusiones — Estructura

- ¿Cuántas variables numéricas hay realmente para trabajar (descontando `country`/`iso_code`/`countryiso3`)?
- ¿El hecho de que sea un panel país-año (144 países × hasta 28 años) cambia algo de cómo pensás las filas del dataset?

<!-- Completar -->

### 2. Nulos por columna

¿Dónde falta información y cuánta? `iso_code`/`countryiso3` quedan afuera del análisis numérico por ser redundantes con `country` (misma cardinalidad, 144 países).

In [ ]:
resumen_nulos = pd.DataFrame({
    'Tipo de dato': df.dtypes.astype(str),
    'No nulos': df.notna().sum(),
    'Nulos': df.isna().sum(),
    '% nulos': (df.isna().sum() / len(df) * 100).round(2),
}).sort_values('% nulos', ascending=False)

# Se muestra sin estilizar (mismo criterio que TP2): mezcla texto + numeros, y lo
# que importa aca es poder ordenar/leer la tabla completa, no resaltarla con color.
display(resumen_nulos)

# Barra horizontal de % de nulos: mas legible que msno.matrix con 64 etiquetas en el eje.
datos_grafico = resumen_nulos[resumen_nulos['% nulos'] > 0].sort_values('% nulos')
fig, ax = plt.subplots(figsize=(8, 14))
ax.barh(datos_grafico.index, datos_grafico['% nulos'], color='#4C72B0')
ax.set_xlabel('% de nulos')
ax.set_title('Proporcion de valores faltantes por columna')
plt.tight_layout()
plt.show()

print(f"Columnas sin nulos: {(resumen_nulos['% nulos'] == 0).sum()} de {len(resumen_nulos)}")

### Conclusiones — Nulos por columna

- Solo 6 de 64 columnas no tienen ningún nulo. ¿Qué tipo de variables son las que más faltantes tienen (índices compuestos, indicadores sociales, ambos)?
- ¿Alguna de las variables con muchísimos nulos (`adult_literacy_rate` 80%, `fiscal_health`/`judicial_effectiveness` ~75%) es imprescindible para lo que sigue, o se puede directamente descartar?

<!-- Completar -->

### 3. Patrón de nulos (ubicación y correlación de ausencia)

¿El patrón de nulos es aleatorio, o hay columnas que faltan juntas? Se dispara la heurística de `msno.matrix`/`msno.heatmap` obligatorios (nulos > 5% en la enorme mayoría de columnas). Se acota a las columnas con más de 5% de nulos (33 de 64) para que el heatmap sea legible en vez de forzar una matriz 64×64.

In [ ]:
columnas_con_nulos = resumen_nulos[resumen_nulos['% nulos'] > 5].index.tolist()

msno.matrix(df[columnas_con_nulos], figsize=(14, 6), fontsize=8)
plt.title('Ubicacion de los nulos (columnas con >5% de faltantes)')
plt.show()

msno.heatmap(df[columnas_con_nulos], figsize=(14, 12), fontsize=8)
plt.title('Correlacion de nulidad entre columnas')
plt.show()

### Conclusiones — Patrón de nulos

- `judicial_effectiveness` y `fiscal_health` tienen correlación de nulidad ≈1, igual que los dos `poverty_headcount_ratio` con `gini_index`. ¿Qué sugiere esto sobre el origen de esos faltantes (una misma encuesta/fuente que no cubre ciertos países-año)?
- ¿Este patrón es más compatible con MCAR (aleatorio) o con MAR/MNAR (asociado a qué tan desarrollado está el país o a qué tan buena es su infraestructura estadística)?

<!-- Completar -->

### 4. Rango y orden de magnitud de cada variable numérica

`describe()` transpuesto y coloreado por columna (cada estadístico se compara contra sí mismo a través de las variables) para que salte a la vista la diferencia de escalas.

In [ ]:
describe_numericas = df[columnas_numericas].describe().T
display(estilizar(describe_numericas, cmap='Blues', precision=2))

### Conclusiones — Rango y magnitud

- ¿Qué grupos de variables comparten escala (por ejemplo, los sub-índices de libertad económica en 0-100) y cuáles están en unidades completamente distintas (USD corrientes, kg de petróleo equivalente per cápita, años de vida)?
- Dada esta disparidad de escalas, ¿qué familias de modelos del resto de la guía (Ejercicio 3 en adelante) van a necesitar estandarización sí o sí?

<!-- Completar -->

### 5. Correlación con el target

Con 60 variables numéricas no tiene sentido graficar la distribución de todas ("principales variables" de la consigna) — se usa la correlación de Pearson con el target como criterio objetivo de selección. De paso, esto anticipa el filtro univariado que el Ejercicio 3 va a aplicar formalmente (no se resuelve acá).

In [ ]:
correlacion_target = df[columnas_numericas].corr(numeric_only=True)[TARGET].drop(TARGET)
correlacion_target = correlacion_target.reindex(correlacion_target.abs().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(8, 14))
colores = ['#4C72B0' if v > 0 else '#C44E52' for v in correlacion_target.values]
ax.barh(correlacion_target.index[::-1], correlacion_target.values[::-1], color=colores[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel(f'Correlacion de Pearson con {TARGET}')
ax.set_title('Correlacion de cada variable numerica con el target')
plt.tight_layout()
plt.show()

# Top 10 por |r|: es el subconjunto que se usa en el punto 6 como "variables principales".
variables_principales = correlacion_target.head(10).index.tolist()
display(correlacion_target.head(10).rename('Correlacion con el target').to_frame())

### Conclusiones — Correlación con el target

- Las variables más correlacionadas son mayormente sub-índices de institucionalidad (`government_integrity`, `property_rights`, `judicial_effectiveness`) más que variables "económicas" directas. ¿Te sorprende, o tiene sentido con lo que sabés de desarrollo económico?
- Todas estas correlaciones son con la escala original del target (`gdp_per_capita_current_us$`, con skew alto). ¿Cambiaría el ranking si se calculara sobre el target transformado (log)? Es una pregunta para cuando llegues al punto 3 de este mismo ejercicio.

<!-- Completar -->

### 6. Distribuciones de las variables principales

Histograma+KDE, boxplot y tabla de skewness/curtosis del target y las 10 variables más correlacionadas con él. Q-Q plot automático en las que tengan \|skew\| > 1 (heurística de asimetría relevante).

In [ ]:
variables_a_graficar = [TARGET] + variables_principales
n_cols = 4
n_filas = int(np.ceil(len(variables_a_graficar) / n_cols))

# Histogramas + KDE
fig, axes = plt.subplots(n_filas, n_cols, figsize=(20, 4 * n_filas))
axes = axes.flatten()
for i, col in enumerate(variables_a_graficar):
  sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='#4C72B0')
  axes[i].set_title(col, fontsize=10)
for j in range(len(variables_a_graficar), len(axes)):
  fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

# Boxplots (misma grilla, para ver outliers y dispersion)
fig, axes = plt.subplots(n_filas, n_cols, figsize=(20, 3 * n_filas))
axes = axes.flatten()
for i, col in enumerate(variables_a_graficar):
  sns.boxplot(x=df[col].dropna(), ax=axes[i], color='#4C72B0')
  axes[i].set_title(col, fontsize=10)
for j in range(len(variables_a_graficar), len(axes)):
  fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

# Tabla de skewness/curtosis
filas_skew = []
for col in variables_a_graficar:
  serie = df[col].dropna()
  filas_skew.append({
      'Variable': col,
      'Skewness': stats.skew(serie),
      'Curtosis': stats.kurtosis(serie),
      '% nulos': df[col].isna().mean() * 100,
  })
tabla_skew = pd.DataFrame(filas_skew).set_index('Variable')
display(estilizar(tabla_skew, cmap='Oranges', precision=2))

# Q-Q plot automatico: heuristica, |skew| > 1 indica asimetria relevante para decisiones posteriores
vars_asimetricas = tabla_skew[tabla_skew['Skewness'].abs() > 1].index.tolist()
if vars_asimetricas:
  fig, axes = plt.subplots(1, len(vars_asimetricas), figsize=(5 * len(vars_asimetricas), 4))
  if len(vars_asimetricas) == 1:
    axes = [axes]
  for ax, col in zip(axes, vars_asimetricas):
    stats.probplot(df[col].dropna(), dist='norm', plot=ax)
    ax.set_title(f'Q-Q: {col}')
  plt.tight_layout()
  plt.show()
  print(f"Q-Q automatico por |skew| > 1 en: {', '.join(vars_asimetricas)}")

### Conclusiones — Distribuciones

- `gdp_per_capita_current_us$` (el target) tiene skew ≈2.3: muy concentrado en valores bajos con una cola larga de países ricos. ¿Qué transformación (punto 3 de este ejercicio) tiene más sentido para esto?
- `energy_use_kg_of_oil_equivalent_per_capita` y `domestic_credit_to_private_sector_of_gdp` también salen asimétricas — ¿por qué transformación pasarías cada una?
- Los sub-índices de institucionalidad (`government_integrity`, `property_rights`, etc.) tienen formas bastante distintas entre sí a pesar de compartir escala 0-100 (algunos multimodales). ¿Qué podría explicar esa multimodalidad?

<!-- Completar -->

### 7. Sub-índices de libertad económica (misma escala 0-100)

Boxplot comparativo — tiene sentido ponerlos en un mismo panel porque comparten unidad, a diferencia de mezclarlos con variables en USD o años de vida.

In [ ]:
indices_libertad_100 = ['heritage_score', 'property_rights', 'government_integrity',
                         'judicial_effectiveness', 'tax_burden', 'government_spending',
                         'business_freedom', 'labor_freedom', 'monetary_freedom',
                         'trade_freedom', 'investment_freedom', 'financial_freedom']
indices_libertad_100 = [c for c in indices_libertad_100 if c in df.columns]

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df[indices_libertad_100], ax=ax, color='#4C72B0')
ax.set_xticks(range(len(indices_libertad_100)))
ax.set_xticklabels(indices_libertad_100, rotation=45, ha='right')
ax.set_title('Sub-indices de libertad economica (todos en escala 0-100)')
plt.tight_layout()
plt.show()

### Conclusiones — Sub-índices de libertad económica

- ¿Cuáles de estos sub-índices tienen mayor dispersión entre países, y cuáles están más concentrados (poca variabilidad, quizás poco útiles para diferenciar países)?
- ¿Hay algún sub-índice con outliers claramente marcados en el boxplot? ¿Coincide con algún país que ya viste en las variables anteriores?

<!-- Completar -->